	# VGG6 on CIFAR-10: Assignment 1

## 1. Environment Setup and Imports

In [43]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import torch.nn.init as init
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torch.optim as optim


## 2. CIFAR-10 Dataset and Data Preprocessing

### **Data Loader**

In [6]:
from PIL import Image, ImageEnhance, ImageOps
import random
import numpy as np
import torch


class Cutout(object):
    """Randomly mask out one or more patches from an image.
    Args:
        n_holes (int): Number of patches to cut out of each image.
        length (int): The length (in pixels) of each square patch.
    """
    def __init__(self, n_holes, length):
        self.n_holes = n_holes
        self.length = length

    def __call__(self, img):
        """
        Args:
            img (Tensor): Tensor image of size (C, H, W).
        Returns:
            Tensor: Image with n_holes of dimension length x length cut out of it.
        """
        h = img.size(1)
        w = img.size(2)

        mask = np.ones((h, w), np.float32)

        for n in range(self.n_holes):
            y = np.random.randint(h)
            x = np.random.randint(w)

            y1 = np.clip(y - self.length // 2, 0, h)
            y2 = np.clip(y + self.length // 2, 0, h)
            x1 = np.clip(x - self.length // 2, 0, w)
            x2 = np.clip(x + self.length // 2, 0, w)

            mask[y1: y2, x1: x2] = 0.

        mask = torch.from_numpy(mask)
        mask = mask.expand_as(img)
        img = img * mask
        return img

class SubPolicy(object):
    def __init__(self, p1, operation1, magnitude_idx1, p2, operation2, magnitude_idx2, fillcolor=(128, 128, 128)):
        self.p1 = p1
        self.op1=operation1
        self.magnitude_idx1=magnitude_idx1
        self.p2 = p2
        self.op2=operation2
        self.magnitude_idx2=magnitude_idx2
        self.fillcolor=fillcolor
        self.init = 0

    def gen(self, operation1, magnitude_idx1, operation2, magnitude_idx2, fillcolor):
        ranges = {
            "shearX": np.linspace(0, 0.3, 10),
            "shearY": np.linspace(0, 0.3, 10),
            "translateX": np.linspace(0, 150 / 331, 10),
            "translateY": np.linspace(0, 150 / 331, 10),
            "rotate": np.linspace(0, 30, 10),
            "color": np.linspace(0.0, 0.9, 10),
            "posterize": np.round(np.linspace(8, 4, 10), 0).astype(int),
            "solarize": np.linspace(256, 0, 10),
            "contrast": np.linspace(0.0, 0.9, 10),
            "sharpness": np.linspace(0.0, 0.9, 10),
            "brightness": np.linspace(0.0, 0.9, 10),
            "autocontrast": [0] * 10,
            "equalize": [0] * 10,
            "invert": [0] * 10
        }
        def rotate_with_fill(img, magnitude):
            rot = img.convert("RGBA").rotate(magnitude)
            return Image.composite(rot, Image.new("RGBA", rot.size, (128,) * 4), rot).convert(img.mode)

        func = {
            "shearX": lambda img, magnitude: img.transform(
                img.size, Image.AFFINE, (1, magnitude *
                                         random.choice([-1, 1]), 0, 0, 1, 0),
                Image.BICUBIC, fillcolor=fillcolor),
            "shearY": lambda img, magnitude: img.transform(
                img.size, Image.AFFINE, (1, 0, 0, magnitude *
                                         random.choice([-1, 1]), 1, 0),
                Image.BICUBIC, fillcolor=fillcolor),
            "translateX": lambda img, magnitude: img.transform(
                img.size, Image.AFFINE, (1, 0, magnitude *
                                         img.size[0] * random.choice([-1, 1]), 0, 1, 0),
                fillcolor=fillcolor),
            "translateY": lambda img, magnitude: img.transform(
                img.size, Image.AFFINE, (1, 0, 0, 0, 1, magnitude *
                                         img.size[1] * random.choice([-1, 1])),
                fillcolor=fillcolor),
            "rotate": lambda img, magnitude: rotate_with_fill(img, magnitude),
            # "rotate": lambda img, magnitude: img.rotate(magnitude * random.choice([-1, 1])),
            "color": lambda img, magnitude: ImageEnhance.Color(img).enhance(1 + magnitude * random.choice([-1, 1])),
            "posterize": lambda img, magnitude: ImageOps.posterize(img, magnitude),
            "solarize": lambda img, magnitude: ImageOps.solarize(img, magnitude),
            "contrast": lambda img, magnitude: ImageEnhance.Contrast(img).enhance(
                1 + magnitude * random.choice([-1, 1])),
            "sharpness": lambda img, magnitude: ImageEnhance.Sharpness(img).enhance(
                1 + magnitude * random.choice([-1, 1])),
            "brightness": lambda img, magnitude: ImageEnhance.Brightness(img).enhance(
                1 + magnitude * random.choice([-1, 1])),
            "autocontrast": lambda img, magnitude: ImageOps.autocontrast(img),
            "equalize": lambda img, magnitude: ImageOps.equalize(img),
            "invert": lambda img, magnitude: ImageOps.invert(img)
        }

        self.operation1 = func[operation1]
        self.magnitude1 = ranges[operation1][magnitude_idx1]
        self.operation2 = func[operation2]
        self.magnitude2 = ranges[operation2][magnitude_idx2]

    def __call__(self, img):
        if self.init == 0:
            self.gen(self.op1, self.magnitude_idx1, self.op2, self.magnitude_idx2, self.fillcolor)
            self.init = 1
        if random.random() < self.p1:
            img = self.operation1(img, self.magnitude1)
        if random.random() < self.p2:
            img = self.operation2(img, self.magnitude2)
        return img

class ImageNetPolicy(object):
    """ Randomly choose one of the best 24 Sub-policies on ImageNet.
        Example:
        >>> policy = ImageNetPolicy()
        >>> transformed = policy(image)
        Example as a PyTorch Transform:
        >>> transform=transforms.Compose([
        >>>     transforms.Resize(256),
        >>>     ImageNetPolicy(),
        >>>     transforms.ToTensor()])
    """

    def __init__(self, fillcolor=(128, 128, 128)):
        self.policies = [
            SubPolicy(0.4, "posterize", 8, 0.6, "rotate", 9, fillcolor),
            SubPolicy(0.6, "solarize", 5, 0.6, "autocontrast", 5, fillcolor),
            SubPolicy(0.8, "equalize", 8, 0.6, "equalize", 3, fillcolor),
            SubPolicy(0.6, "posterize", 7, 0.6, "posterize", 6, fillcolor),
            SubPolicy(0.4, "equalize", 7, 0.2, "solarize", 4, fillcolor),

            SubPolicy(0.4, "equalize", 4, 0.8, "rotate", 8, fillcolor),
            SubPolicy(0.6, "solarize", 3, 0.6, "equalize", 7, fillcolor),
            SubPolicy(0.8, "posterize", 5, 1.0, "equalize", 2, fillcolor),
            SubPolicy(0.2, "rotate", 3, 0.6, "solarize", 8, fillcolor),
            SubPolicy(0.6, "equalize", 8, 0.4, "posterize", 6, fillcolor),

            SubPolicy(0.8, "rotate", 8, 0.4, "color", 0, fillcolor),
            SubPolicy(0.4, "rotate", 9, 0.6, "equalize", 2, fillcolor),
            SubPolicy(0.0, "equalize", 7, 0.8, "equalize", 8, fillcolor),
            SubPolicy(0.6, "invert", 4, 1.0, "equalize", 8, fillcolor),
            SubPolicy(0.6, "color", 4, 1.0, "contrast", 8, fillcolor),

            SubPolicy(0.8, "rotate", 8, 1.0, "color", 2, fillcolor),
            SubPolicy(0.8, "color", 8, 0.8, "solarize", 7, fillcolor),
            SubPolicy(0.4, "sharpness", 7, 0.6, "invert", 8, fillcolor),
            SubPolicy(0.6, "shearX", 5, 1.0, "equalize", 9, fillcolor),
            SubPolicy(0.4, "color", 0, 0.6, "equalize", 3, fillcolor),

            SubPolicy(0.4, "equalize", 7, 0.2, "solarize", 4, fillcolor),
            SubPolicy(0.6, "solarize", 5, 0.6, "autocontrast", 5, fillcolor),
            SubPolicy(0.6, "invert", 4, 1.0, "equalize", 8, fillcolor),
            SubPolicy(0.6, "color", 4, 1.0, "contrast", 8, fillcolor)
        ]

    def __call__(self, img):
        policy_idx = random.randint(0, len(self.policies) - 1)
        return self.policies[policy_idx](img)

    def __repr__(self):
        return "AutoAugment ImageNet Policy"

class CIFAR10Policy(object):
    """ Randomly choose one of the best 25 Sub-policies on CIFAR10.

        Example:
        >>> policy = CIFAR10Policy()
        >>> transformed = policy(image)

        Example as a PyTorch Transform:
        >>> transform=transforms.Compose([
        >>>     transforms.Resize(256),
        >>>     CIFAR10Policy(),
        >>>     transforms.ToTensor()])
    """

    def __init__(self, fillcolor=(128, 128, 128)):
        self.policies = [
            SubPolicy(0.1, "invert", 7, 0.2, "contrast", 6, fillcolor),
            SubPolicy(0.7, "rotate", 2, 0.3, "translateX", 9, fillcolor),
            SubPolicy(0.8, "sharpness", 1, 0.9, "sharpness", 3, fillcolor),
            SubPolicy(0.5, "shearY", 8, 0.7, "translateY", 9, fillcolor),
            SubPolicy(0.5, "autocontrast", 8, 0.9, "equalize", 2, fillcolor),

            SubPolicy(0.2, "shearY", 7, 0.3, "posterize", 7, fillcolor),
            SubPolicy(0.4, "color", 3, 0.6, "brightness", 7, fillcolor),
            SubPolicy(0.3, "sharpness", 9, 0.7, "brightness", 9, fillcolor),
            SubPolicy(0.6, "equalize", 5, 0.5, "equalize", 1, fillcolor),
            SubPolicy(0.6, "contrast", 7, 0.6, "sharpness", 5, fillcolor),

            SubPolicy(0.7, "color", 7, 0.5, "translateX", 8, fillcolor),
            SubPolicy(0.3, "equalize", 7, 0.4, "autocontrast", 8, fillcolor),
            SubPolicy(0.4, "translateY", 3, 0.2, "sharpness", 6, fillcolor),
            SubPolicy(0.9, "brightness", 6, 0.2, "color", 8, fillcolor),
            SubPolicy(0.5, "solarize", 2, 0.0, "invert", 3, fillcolor),

            SubPolicy(0.2, "equalize", 0, 0.6, "autocontrast", 0, fillcolor),
            SubPolicy(0.2, "equalize", 8, 0.8, "equalize", 4, fillcolor),
            SubPolicy(0.9, "color", 9, 0.6, "equalize", 6, fillcolor),
            SubPolicy(0.8, "autocontrast", 4, 0.2, "solarize", 8, fillcolor),
            SubPolicy(0.1, "brightness", 3, 0.7, "color", 0, fillcolor),

            SubPolicy(0.4, "solarize", 5, 0.9, "autocontrast", 3, fillcolor),
            SubPolicy(0.9, "translateY", 9, 0.7, "translateY", 9, fillcolor),
            SubPolicy(0.9, "autocontrast", 2, 0.8, "solarize", 3, fillcolor),
            SubPolicy(0.8, "equalize", 8, 0.1, "invert", 3, fillcolor),
            SubPolicy(0.7, "translateY", 9, 0.9, "autocontrast", 1, fillcolor)
        ]

    def __call__(self, img):
        policy_idx = random.randint(0, len(self.policies) - 1)
        return self.policies[policy_idx](img)

    def __repr__(self):
        return "AutoAugment CIFAR10 Policy"


In [7]:
def GetCifar10(batchsize):
    trans_t = transforms.Compose([transforms.RandomCrop(32, padding=4),
                                  transforms.RandomHorizontalFlip(),
                                  # CIFAR10Policy(),
                                  transforms.ToTensor(),
                                  transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
                                  Cutout(n_holes=1, length=16)
                                  ])
    trans = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))])
    train_data = datasets.CIFAR10('./data', train=True, transform=trans_t, download=True)
    test_data = datasets.CIFAR10('./data', train=False, transform=trans, download=True)
    train_dataloader = DataLoader(train_data, batch_size=batchsize, shuffle=True, num_workers=2)
    test_dataloader = DataLoader(test_data, batch_size=batchsize, shuffle=False, num_workers=2)
    return train_dataloader, test_dataloader

## 3. VGG6 Model Architecture

In [9]:
class VGG(nn.Module):
    def __init__(self, features, num_classes=10):
        super(VGG, self).__init__()
        self.features = features
        self.classifier = nn.Sequential(
            nn.Linear(128, num_classes),
        )
        self._initialize_weights()

    def forward(self, x):
        x = self.features(x)
        x = F.adaptive_avg_pool2d(x, (1, 1))  # fixed output size
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                n = m.kernel_size[0] * m.kernel_size[1] * m.out_channels
                m.weight.data.normal_(0, math.sqrt(2. / n))
                if m.bias is not None:
                    m.bias.data.zero_()
            elif isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                if m.bias is not None:
                    m.bias.data.zero_()
            elif isinstance(m, nn.BatchNorm2d):
                m.weight.data.fill_(1)
                m.bias.data.zero_()

def activation_function(activation):
    if activation == "relu":
      return nn.ReLU(inplace=True)
    elif activation == "sigmoid":
      return nn.Sigmoid()
    elif activation == "tanh":
      return nn.Tanh()
    elif activation == "silu":
      return nn.SiLU(inplace=True)
    elif activation == "gelu":
      return nn.GELU()
    else:
      raise ValueError(f"Unsupported activation: {activation}")

def make_layers(cfg, batch_norm=False, activation="relu"):
    layers = []
    in_channels = 3
    for v in cfg:
        if v == 'M':
            layers += [nn.MaxPool2d(kernel_size=2, stride=2)]
        else:
            conv2d = nn.Conv2d(in_channels, v, kernel_size=3, padding=1)
            if batch_norm:
                layers += [conv2d, nn.BatchNorm2d(v), activation_function(activation)]
            else:
                layers += [conv2d, activation_function(activation)]
            in_channels = v
    return nn.Sequential(*layers)

def vgg(cfg, num_classes=10, batch_norm=True, activation="relu"):
    return VGG(
        make_layers(cfg, batch_norm=batch_norm, activation=activation),
        num_classes=num_classes
    )

# VGG-6 configuration
cfg_vgg6 = [64, 64, 'M', 128, 128, 'M']

model = vgg(cfg_vgg6, num_classes=10, batch_norm=True)
print(model)


VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU(inplace=True)
    (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (7): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (9): ReLU(inplace=True)
    (10): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (12): ReLU(inplace=True)
    (13): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Seque

### 3.1 Activation Function Initialization Checks

In [10]:
for activation in ["relu", "sigmoid", "tanh", "silu", "gelu"]:
    test_model = vgg(
        cfg_vgg6,
        num_classes=10,
        batch_norm=True,
        activation=activation
    )

    print(activation, "created successfully")

relu created successfully
sigmoid created successfully
tanh created successfully
silu created successfully
gelu created successfully


## 4. Training and Evaluation Functions

In [11]:
def evaluate(model, dataloader, criterion, device):
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)

            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    avg_loss = running_loss / total
    accuracy = 100.0 * correct / total

    return avg_loss, accuracy

In [12]:
train_loader,test_loader = GetCifar10(64)

100%|██████████| 170M/170M [00:28<00:00, 6.09MB/s]


In [13]:
print([name for name in globals() if "loader" in name.lower()])
print("Training batches:", len(train_loader))
print("Testing batches:", len(test_loader))

images, labels = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Image data type:", images.dtype)
print("Label data type:", labels.dtype)

['__loader__', 'DataLoader', 'train_loader', 'test_loader']
Training batches: 782
Testing batches: 157
Image batch shape: torch.Size([64, 3, 32, 32])
Label batch shape: torch.Size([64])
Image data type: torch.float32
Label data type: torch.int64


### Training and Validation Split

The CIFAR-10 training dataset contains 50,000 images. It is divided into 45,000 training images and 5,000 validation images using a fixed random seed of 42.

Training images use data augmentation, while validation images use normalization only. The official CIFAR-10 test dataset is kept separate for final evaluation.

In [14]:
import torchvision
from torchvision import transforms
from torch.utils.data import Subset, DataLoader
import numpy as np

# Reproducible split
np.random.seed(42)

# Redefine transforms locally since they were defined inside GetCifar10
trans_t = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    Cutout(n_holes=1, length=16)
])

trans = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

# Create separate datasets with different transforms
train_dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=trans_t
)

val_dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=trans
)

# Shuffle indices and split
indices = np.random.permutation(len(train_dataset))

train_indices = indices[:45000]
val_indices = indices[45000:]

# Create subsets
train_subset = Subset(train_dataset, train_indices)
val_subset = Subset(val_dataset, val_indices)

# Create DataLoaders
train_loader = DataLoader(
    train_subset,
    batch_size=64,
    shuffle=True,
    num_workers=2
)

val_loader = DataLoader(
    val_subset,
    batch_size=64,
    shuffle=False,
    num_workers=2
)

print("Training images:", len(train_subset))
print("Validation images:", len(val_subset))
print("Test images:", len(test_loader.dataset))

Training images: 45000
Validation images: 5000
Test images: 10000


### 4.1 Device Configuration and Model Initialization

In [15]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = vgg(cfg_vgg6, num_classes=10, batch_norm=True).to(device)

print("Device:", device)
print("Model:", type(model).__name__)
print("Parameters:", sum(p.numel() for p in model.parameters()))

Device: cuda
Model: VGG
Parameters: 262218


In [16]:
import torch.nn as nn
import torch.optim as optim

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

print("Loss function:", criterion)
print("Optimizer:", optimizer)

Loss function: CrossEntropyLoss()
Optimizer: Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)


In [17]:
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in dataloader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    avg_loss = running_loss / total
    accuracy = 100.0 * correct / total

    return avg_loss, accuracy

In [ ]:
train_loss, train_acc = train_one_epoch(
    model, train_loader, criterion, optimizer, device
)

val_loss, val_acc = evaluate(
    model, val_loader, criterion, device
)

print(f"Training Loss: {train_loss:.4f}")
print(f"Training Accuracy: {train_acc:.2f}%")
print(f"Validation Loss: {val_loss:.4f}")
print(f"Validation Accuracy: {val_acc:.2f}%")

Training Loss: 1.5238
Training Accuracy: 44.77%
Validation Loss: 1.5394
Validation Accuracy: 48.48%


In [2]:
!pip install -q wandb
import wandb

wandb.login()

/usr/local/lib/python3.13/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: cs25m518 (cs25m518-indian-institute-of-technology-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## 5. Q1: Baseline VGG6 Training and Evaluation

In [ ]:
import wandb
import torch
import torch.nn as nn
import torch.optim as optim

# Experiment configuration
config = {
    "model": "VGG6",
    "activation": "ReLU",
    "optimizer": "Adam",
    "learning_rate": 0.001,
    "batch_size": 64,
    "epochs": 10,
    "seed": 42
}

# Reproducibility
torch.manual_seed(config["seed"])
torch.cuda.manual_seed_all(config["seed"])

# Fresh model
device = torch.device("cuda")
model = vgg(cfg_vgg6, num_classes=10, batch_norm=True).to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=config["learning_rate"]
)

# Initialize W&B
wandb.init(
    project="VGG6-CIFAR10-Assignment1",
    name="baseline-ReLU-Adam",
    config=config
)

print("Baseline experiment initialized!")

Baseline experiment initialized!


In [ ]:
best_val_acc = 0.0

for epoch in range(config["epochs"]):

    train_loss, train_acc = train_one_epoch(
        model, train_loader, criterion, optimizer, device
    )

    val_loss, val_acc = evaluate(
        model, val_loader, criterion, device
    )

    wandb.log({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_accuracy": train_acc,
        "val_loss": val_loss,
        "val_accuracy": val_acc
    })

    print(
        f"Epoch [{epoch+1}/{config['epochs']}] "
        f"Train Loss: {train_loss:.4f} "
        f"Train Acc: {train_acc:.2f}% "
        f"Val Loss: {val_loss:.4f} "
        f"Val Acc: {val_acc:.2f}%"
    )

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "best_vgg6_baseline.pth")

print("Best validation accuracy:", best_val_acc)

wandb.finish()

Epoch [1/10] Train Loss: 1.5293 Train Acc: 44.26% Val Loss: 1.5018 Val Acc: 48.60%
Epoch [2/10] Train Loss: 1.2548 Train Acc: 55.22% Val Loss: 1.3013 Val Acc: 56.34%
Epoch [3/10] Train Loss: 1.1277 Train Acc: 60.24% Val Loss: 1.5638 Val Acc: 52.82%
Epoch [4/10] Train Loss: 1.0410 Train Acc: 63.49% Val Loss: 1.2136 Val Acc: 61.38%
Epoch [5/10] Train Loss: 0.9795 Train Acc: 65.49% Val Loss: 1.0529 Val Acc: 65.14%
Epoch [6/10] Train Loss: 0.9281 Train Acc: 67.50% Val Loss: 1.0269 Val Acc: 66.84%
Epoch [7/10] Train Loss: 0.8916 Train Acc: 68.87% Val Loss: 1.1144 Val Acc: 65.82%
Epoch [8/10] Train Loss: 0.8567 Train Acc: 70.02% Val Loss: 0.8461 Val Acc: 71.04%
Epoch [9/10] Train Loss: 0.8279 Train Acc: 71.02% Val Loss: 0.7400 Val Acc: 75.08%
Epoch [10/10] Train Loss: 0.7999 Train Acc: 71.98% Val Loss: 0.8755 Val Acc: 71.14%
Best validation accuracy: 75.08


epoch,▁▂▃▃▄▅▆▆▇█
train_accuracy,▁▄▅▆▆▇▇███
train_loss,█▅▄▃▃▂▂▂▁▁
val_accuracy,▁▃▂▄▅▆▆▇█▇
val_loss,▇▆█▅▄▃▄▂▁▂
epoch,10
train_accuracy,71.98444
train_loss,0.79992
val_accuracy,71.14
val_loss,0.87547


In [ ]:
 # Load the best validation checkpoint
model.load_state_dict(
    torch.load("best_vgg6_baseline.pth", map_location=device)
)

# Evaluate on the untouched test set
test_loss, test_acc = evaluate(
    model, test_loader, criterion, device
)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Top-1 Accuracy: {test_acc:.2f}%")

Test Loss: 0.7438
Test Top-1 Accuracy: 74.98%


In [ ]:
from google.colab import files
files.download("best_vgg6_baseline.pth")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
test_model = vgg(
    cfg_vgg6,
    num_classes=10,
    batch_norm=True,
    activation="gelu"
).to(device)

images, labels = next(iter(train_loader))
images = images.to(device)

outputs = test_model(images)

print("Input shape:", images.shape)
print("Output shape:", outputs.shape)

Input shape: torch.Size([64, 3, 32, 32])
Output shape: torch.Size([64, 10])


## 6. Q2(a): Comparison of Activation Functions

In [ ]:
import wandb
import torch
import torch.nn as nn
import torch.optim as optim

activations = ["relu", "sigmoid", "tanh", "silu", "gelu"]

activation_results = []

for activation in activations:

    print(f"\nStarting experiment: {activation.upper()}")

    # Fresh model for each experiment
    model = vgg(
        cfg_vgg6,
        num_classes=10,
        batch_norm=True,
        activation=activation
    ).to(device)

    criterion = nn.CrossEntropyLoss()

    optimizer = optim.Adam(
        model.parameters(),
        lr=0.001
    )

    best_val_acc = 0.0

    run = wandb.init(
        project="VGG6-CIFAR10-Assignment1",
        name=f"activation-{activation}",
        config={
            "model": "VGG6",
            "activation": activation,
            "optimizer": "Adam",
            "learning_rate": 0.001,
            "batch_size": 64,
            "epochs": 10,
            "seed": 42
        },
        reinit=True
    )

    for epoch in range(10):

        train_loss, train_acc = train_one_epoch(
            model, train_loader, criterion, optimizer, device
        )

        val_loss, val_acc = evaluate(
            model, val_loader, criterion, device
        )

        wandb.log({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "train_accuracy": train_acc,
            "val_loss": val_loss,
            "val_accuracy": val_acc
        })

        print(
            f"{activation.upper()} | "
            f"Epoch {epoch+1}/10 | "
            f"Train Acc: {train_acc:.2f}% | "
            f"Val Acc: {val_acc:.2f}%"
        )

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(
                model.state_dict(),
                f"best_vgg6_{activation}.pth"
            )

    activation_results.append({
        "activation": activation,
        "best_val_accuracy": best_val_acc
    })

    wandb.summary["best_val_accuracy"] = best_val_acc
    wandb.finish()

print("\nACTIVATION EXPERIMENT SUMMARY")
for result in activation_results:
    print(result)


Starting experiment: RELU


wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


RELU | Epoch 1/10 | Train Acc: 44.49% | Val Acc: 48.24%
RELU | Epoch 2/10 | Train Acc: 55.33% | Val Acc: 55.00%
RELU | Epoch 3/10 | Train Acc: 59.89% | Val Acc: 53.60%
RELU | Epoch 4/10 | Train Acc: 62.95% | Val Acc: 60.66%
RELU | Epoch 5/10 | Train Acc: 65.44% | Val Acc: 63.20%
RELU | Epoch 6/10 | Train Acc: 67.38% | Val Acc: 68.36%
RELU | Epoch 7/10 | Train Acc: 68.72% | Val Acc: 69.42%
RELU | Epoch 8/10 | Train Acc: 69.96% | Val Acc: 71.54%
RELU | Epoch 9/10 | Train Acc: 70.84% | Val Acc: 73.12%
RELU | Epoch 10/10 | Train Acc: 71.86% | Val Acc: 73.84%


epoch,▁▂▃▃▄▅▆▆▇█
train_accuracy,▁▄▅▆▆▇▇███
train_loss,█▅▄▃▃▂▂▂▁▁
val_accuracy,▁▃▂▄▅▇▇▇██
val_loss,█▆▇▅▅▂▂▂▁▁
best_val_accuracy,73.84
epoch,10
train_accuracy,71.85556
train_loss,0.80518
val_accuracy,73.84
val_loss,0.79576



Starting experiment: SIGMOID


SIGMOID | Epoch 1/10 | Train Acc: 32.36% | Val Acc: 36.06%
SIGMOID | Epoch 2/10 | Train Acc: 41.79% | Val Acc: 35.34%
SIGMOID | Epoch 3/10 | Train Acc: 45.78% | Val Acc: 18.14%
SIGMOID | Epoch 4/10 | Train Acc: 49.00% | Val Acc: 30.92%
SIGMOID | Epoch 5/10 | Train Acc: 51.31% | Val Acc: 12.14%
SIGMOID | Epoch 6/10 | Train Acc: 52.99% | Val Acc: 31.76%
SIGMOID | Epoch 7/10 | Train Acc: 54.79% | Val Acc: 30.26%
SIGMOID | Epoch 8/10 | Train Acc: 56.08% | Val Acc: 35.20%
SIGMOID | Epoch 9/10 | Train Acc: 57.25% | Val Acc: 38.24%
SIGMOID | Epoch 10/10 | Train Acc: 57.80% | Val Acc: 33.88%


... (7 outputs left out)


Starting experiment: GELU


GELU | Epoch 1/10 | Train Acc: 45.35% | Val Acc: 53.82%
GELU | Epoch 2/10 | Train Acc: 56.43% | Val Acc: 58.24%
GELU | Epoch 3/10 | Train Acc: 61.14% | Val Acc: 62.94%
GELU | Epoch 4/10 | Train Acc: 64.38% | Val Acc: 55.04%
GELU | Epoch 5/10 | Train Acc: 66.48% | Val Acc: 71.24%
GELU | Epoch 6/10 | Train Acc: 67.98% | Val Acc: 69.94%
GELU | Epoch 7/10 | Train Acc: 70.00% | Val Acc: 70.76%
GELU | Epoch 8/10 | Train Acc: 70.89% | Val Acc: 72.22%
GELU | Epoch 9/10 | Train Acc: 71.84% | Val Acc: 73.96%
GELU | Epoch 10/10 | Train Acc: 72.63% | Val Acc: 76.72%


epoch,▁▂▃▃▄▅▆▆▇█
train_accuracy,▁▄▅▆▆▇▇███
train_loss,█▅▄▃▃▂▂▁▁▁
val_accuracy,▁▂▄▁▆▆▆▇▇█
val_loss,▇▆▄█▂▃▃▂▂▁
best_val_accuracy,76.72
epoch,10
train_accuracy,72.62667
train_loss,0.78213
val_accuracy,76.72
val_loss,0.69264



ACTIVATION EXPERIMENT SUMMARY
{'activation': 'relu', 'best_val_accuracy': 73.84}
{'activation': 'sigmoid', 'best_val_accuracy': 38.24}
{'activation': 'tanh', 'best_val_accuracy': 63.82}
{'activation': 'silu', 'best_val_accuracy': 74.3}
{'activation': 'gelu', 'best_val_accuracy': 76.72}


In [ ]:
import pandas as pd

df = pd.DataFrame(activation_results)
df = df.sort_values(
    "best_val_accuracy",
    ascending=False
)

display(df)
df.to_csv("activation_results.csv", index=False)
from google.colab import files
files.download("activation_results.csv")

,activation,best_val_accuracy
4,gelu,76.72
3,silu,74.30
0,relu,73.84
2,tanh,63.82
1,sigmoid,38.24


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 7. Q2(b): Comparison of Optimizers

In [20]:
import torch.optim as optim

def get_optimizer(name, model, learning_rate=0.001):
    if name == "SGD":
        return optim.SGD(model.parameters(), lr=learning_rate)

    elif name == "Nesterov-SGD":
        return optim.SGD(
            model.parameters(),
            lr=learning_rate,
            momentum=0.9,
            nesterov=True
        )

    elif name == "Adam":
        return optim.Adam(model.parameters(), lr=learning_rate)

    elif name == "Adagrad":
        return optim.Adagrad(model.parameters(), lr=learning_rate)

    elif name == "RMSprop":
        return optim.RMSprop(model.parameters(), lr=learning_rate)

    elif name == "Nadam":
        return optim.NAdam(model.parameters(), lr=learning_rate)

    else:
        raise ValueError(f"Unknown optimizer: {name}")

In [24]:
import random
import numpy as np
import torch
import wandb
import pandas as pd

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


optimizer_names = [
    "SGD",
    "Nesterov-SGD",
    "Adam",
    "Adagrad",
    "RMSprop",
    "Nadam"
]

optimizer_results = []

for optimizer_name in optimizer_names:

    print(f"\nStarting experiment: {optimizer_name}")

    set_seed(42)

    model = vgg(
        cfg_vgg6,
        num_classes=10,
        batch_norm=True,
        activation="relu"
    ).to(device)

    criterion = torch.nn.CrossEntropyLoss()

    optimizer = get_optimizer(
        optimizer_name,
        model,
        learning_rate=0.001
    )

    run = wandb.init(
        project="VGG6-CIFAR10-Assignment1",
        entity="cs25m518-indian-institute-of-technology-madras",
        name=f"optimizer-{optimizer_name}",
        config={
            "experiment": "optimizer_comparison",
            "optimizer": optimizer_name,
            "activation": "ReLU",
            "learning_rate": 0.001,
            "batch_size": 64,
            "epochs": 10,
            "seed": 42
        }
    )

    best_val_accuracy = 0.0
    best_epoch = 0

    for epoch in range(10):

        train_loss, train_accuracy = train_one_epoch(
            model, train_loader, criterion, optimizer, device
        )

        val_loss, val_accuracy = evaluate(
            model, val_loader, criterion, device
        )

        if val_accuracy > best_val_accuracy:
            best_val_accuracy = val_accuracy
            best_epoch = epoch + 1

        wandb.log({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "train_accuracy": train_accuracy,
            "val_loss": val_loss,
            "val_accuracy": val_accuracy,
            "best_val_accuracy": best_val_accuracy
        })

        print(
            f"Epoch {epoch+1}/10 | "
            f"Train Acc: {train_accuracy:.2f}% | "
            f"Val Acc: {val_accuracy:.2f}%"
        )

    optimizer_results.append({
        "optimizer": optimizer_name,
        "best_val_accuracy": best_val_accuracy,
        "best_epoch": best_epoch
    })

    wandb.finish()

optimizer_df = pd.DataFrame(optimizer_results)
optimizer_df = optimizer_df.sort_values(
    "best_val_accuracy", ascending=False
)

display(optimizer_df)
optimizer_df.to_csv("optimizer_results.csv", index=False)


Starting experiment: SGD


Epoch 1/10 | Train Acc: 23.70% | Val Acc: 28.74%
Epoch 2/10 | Train Acc: 30.37% | Val Acc: 31.12%
Epoch 3/10 | Train Acc: 32.79% | Val Acc: 34.06%
Epoch 4/10 | Train Acc: 34.73% | Val Acc: 35.04%
Epoch 5/10 | Train Acc: 36.40% | Val Acc: 37.50%
Epoch 6/10 | Train Acc: 38.05% | Val Acc: 39.08%
Epoch 7/10 | Train Acc: 38.78% | Val Acc: 40.82%
Epoch 8/10 | Train Acc: 40.08% | Val Acc: 41.70%
Epoch 9/10 | Train Acc: 41.10% | Val Acc: 43.08%
Epoch 10/10 | Train Acc: 42.22% | Val Acc: 42.70%


best_val_accuracy,▁▂▄▄▅▆▇▇██
epoch,▁▂▃▃▄▅▆▆▇█
train_accuracy,▁▄▄▅▆▆▇▇██
train_loss,█▅▄▄▃▃▂▂▁▁
val_accuracy,▁▂▄▄▅▆▇▇██
val_loss,█▆▅▄▃▃▂▂▁▁
best_val_accuracy,43.08
epoch,10
train_accuracy,42.21778
train_loss,1.63257
val_accuracy,42.7



Starting experiment: Nesterov-SGD


Epoch 1/10 | Train Acc: 33.50% | Val Acc: 39.50%
Epoch 2/10 | Train Acc: 42.94% | Val Acc: 43.86%
Epoch 3/10 | Train Acc: 46.57% | Val Acc: 45.82%
Epoch 4/10 | Train Acc: 49.48% | Val Acc: 52.32%
Epoch 5/10 | Train Acc: 51.04% | Val Acc: 54.34%
Epoch 6/10 | Train Acc: 52.95% | Val Acc: 55.34%
Epoch 7/10 | Train Acc: 54.03% | Val Acc: 55.40%
Epoch 8/10 | Train Acc: 55.18% | Val Acc: 57.90%
Epoch 9/10 | Train Acc: 56.28% | Val Acc: 58.92%
Epoch 10/10 | Train Acc: 57.18% | Val Acc: 60.44%


best_val_accuracy,▁▂▃▅▆▆▆▇▇█
epoch,▁▂▃▃▄▅▆▆▇█
train_accuracy,▁▄▅▆▆▇▇▇██
train_loss,█▅▄▃▃▂▂▂▁▁
val_accuracy,▁▂▃▅▆▆▆▇▇█
val_loss,█▆▆▄▃▃▃▂▁▁
best_val_accuracy,60.44
epoch,10
train_accuracy,57.17778
train_loss,1.21011
val_accuracy,60.44


... (9 outputs left out)


Starting experiment: Nadam


Epoch 1/10 | Train Acc: 45.62% | Val Acc: 28.78%
Epoch 2/10 | Train Acc: 56.74% | Val Acc: 46.00%
Epoch 3/10 | Train Acc: 61.67% | Val Acc: 48.14%
Epoch 4/10 | Train Acc: 64.81% | Val Acc: 51.64%
Epoch 5/10 | Train Acc: 67.01% | Val Acc: 63.96%
Epoch 6/10 | Train Acc: 68.45% | Val Acc: 59.28%
Epoch 7/10 | Train Acc: 69.93% | Val Acc: 63.52%
Epoch 8/10 | Train Acc: 71.10% | Val Acc: 46.72%
Epoch 9/10 | Train Acc: 71.66% | Val Acc: 71.10%
Epoch 10/10 | Train Acc: 72.76% | Val Acc: 74.18%


best_val_accuracy,▁▄▄▅▆▆▆▆██
epoch,▁▂▃▃▄▅▆▆▇█
train_accuracy,▁▄▅▆▇▇▇███
train_loss,█▅▄▃▃▂▂▁▁▁
val_accuracy,▁▄▄▅▆▆▆▄██
val_loss,█▃▄▃▂▂▂▆▁▁
best_val_accuracy,74.18
epoch,10
train_accuracy,72.76
train_loss,0.78361
val_accuracy,74.18


,optimizer,best_val_accuracy,best_epoch
2,Adam,75.60,9
5,Nadam,74.18,10
4,RMSprop,66.30,10
1,Nesterov-SGD,60.44,10
3,Adagrad,55.42,8
0,SGD,43.08,9


In [25]:
from google.colab import files
files.download("optimizer_results.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [26]:
from torch.utils.data import DataLoader

def create_loaders(batch_size):
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=2,
        pin_memory=True
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )

    return train_loader, val_loader

## 8. Q2(c): Hyperparameter Experiments

### 8.1 Batch Size Comparison

In [27]:
train_loader_32, val_loader_32 = create_loaders(32)
train_loader_128, val_loader_128 = create_loaders(128)

print("Batch 32:", len(train_loader_32), "training batches")
print("Batch 128:", len(train_loader_128), "training batches")

Batch 32: 1563 training batches
Batch 128: 391 training batches


In [30]:
wandb.init(
    project="VGG6-CIFAR10-Assignment1",
    entity="cs25m518-indian-institute-of-technology-madras",
    name="batch-size-32",
    config={
        "activation": "ReLU",
        "optimizer": "Adam",
        "learning_rate": 0.001,
        "batch_size": 32,
        "epochs": 10,
        "seed": 42
    }
)

set_seed(42)

model = vgg(
    cfg_vgg6, num_classes=10,
    batch_norm=True, activation="relu"
).to(device)

optimizer = get_optimizer("Adam", model, learning_rate=0.001)
criterion = nn.CrossEntropyLoss()

for epoch in range(10):
    train_loss, train_acc = train_one_epoch(
        model, train_loader_32, criterion, optimizer, device
    )

    val_loss, val_acc = evaluate(
        model, val_loader_32, criterion, device
    )

    wandb.log({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_accuracy": train_acc,
        "val_loss": val_loss,
        "val_accuracy": val_acc
    })

    print(
        f"Epoch {epoch+1}/10 | "
        f"Train Acc: {train_acc:.2f}% | "
        f"Val Acc: {val_acc:.2f}%"
    )

wandb.finish()

Epoch 1/10 | Train Acc: 45.52% | Val Acc: 54.33%
Epoch 2/10 | Train Acc: 56.58% | Val Acc: 61.58%
Epoch 3/10 | Train Acc: 61.36% | Val Acc: 61.88%
Epoch 4/10 | Train Acc: 64.71% | Val Acc: 69.80%
Epoch 5/10 | Train Acc: 66.98% | Val Acc: 71.74%
Epoch 6/10 | Train Acc: 68.77% | Val Acc: 71.78%
Epoch 7/10 | Train Acc: 70.17% | Val Acc: 74.11%
Epoch 8/10 | Train Acc: 71.31% | Val Acc: 75.28%
Epoch 9/10 | Train Acc: 72.58% | Val Acc: 78.99%
Epoch 10/10 | Train Acc: 73.30% | Val Acc: 78.19%


epoch,▁▂▃▃▄▅▆▆▇█
train_accuracy,▁▄▅▆▆▇▇▇██
train_loss,█▅▄▃▃▂▂▂▁▁
val_accuracy,▁▃▃▅▆▆▇▇██
val_loss,█▆▆▄▃▃▂▂▁▁
epoch,10
train_accuracy,73.296
train_loss,0.76776
val_accuracy,78.19
val_loss,0.64236


### 8.2 Learning Rate Comparison

In [32]:
train_loader_64, val_loader_64 = create_loaders(64)
wandb.init(
    project="VGG6-CIFAR10-Assignment1",
    entity="cs25m518-indian-institute-of-technology-madras",
    name="learning-rate-0.0001",
    config={
        "activation": "ReLU",
        "optimizer": "Adam",
        "learning_rate": 0.0001,
        "batch_size": 64,
        "epochs": 10,
        "seed": 42
    }
)

set_seed(42)

model = vgg(
    cfg_vgg6, num_classes=10,
    batch_norm=True, activation="relu"
).to(device)

optimizer = get_optimizer("Adam", model, learning_rate=0.0001)
criterion = nn.CrossEntropyLoss()

for epoch in range(10):
    train_loss, train_acc = train_one_epoch(
        model, train_loader_64, criterion, optimizer, device
    )

    val_loss, val_acc = evaluate(
        model, val_loader_64, criterion, device
    )

    wandb.log({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_accuracy": train_acc,
        "val_loss": val_loss,
        "val_accuracy": val_acc
    })

    print(
        f"Epoch {epoch+1}/10 | "
        f"Train Acc: {train_acc:.2f}% | "
        f"Val Acc: {val_acc:.2f}%"
    )

wandb.finish()

Epoch 1/10 | Train Acc: 38.39% | Val Acc: 48.38%
Epoch 2/10 | Train Acc: 48.28% | Val Acc: 53.59%
Epoch 3/10 | Train Acc: 51.57% | Val Acc: 56.14%
Epoch 4/10 | Train Acc: 54.11% | Val Acc: 56.11%
Epoch 5/10 | Train Acc: 55.89% | Val Acc: 60.06%
Epoch 6/10 | Train Acc: 57.09% | Val Acc: 59.42%
Epoch 7/10 | Train Acc: 58.63% | Val Acc: 62.40%
Epoch 8/10 | Train Acc: 59.76% | Val Acc: 63.66%
Epoch 9/10 | Train Acc: 60.67% | Val Acc: 59.35%
Epoch 10/10 | Train Acc: 61.50% | Val Acc: 66.09%


epoch,▁▂▃▃▄▅▆▆▇█
train_accuracy,▁▄▅▆▆▇▇▇██
train_loss,█▅▄▃▃▂▂▂▁▁
val_accuracy,▁▃▄▄▆▅▇▇▅█
val_loss,█▆▅▅▃▄▂▂▄▁
epoch,10
train_accuracy,61.498
train_loss,1.10068
val_accuracy,66.092
val_loss,0.9738


### 8.3 Number of Epochs Comparison

In [33]:
train_loader_64, val_loader_64 = create_loaders(64)
wandb.init(
    project="VGG6-CIFAR10-Assignment1",
    entity="cs25m518-indian-institute-of-technology-madras",
    name="learning-rate-0.01",
    config={
        "activation": "ReLU",
        "optimizer": "Adam",
        "learning_rate": 0.01,
        "batch_size": 64,
        "epochs": 10,
        "seed": 42
    }
)

set_seed(42)

model = vgg(
    cfg_vgg6, num_classes=10,
    batch_norm=True, activation="relu"
).to(device)

optimizer = get_optimizer("Adam", model, learning_rate=0.01)
criterion = nn.CrossEntropyLoss()

for epoch in range(10):
    train_loss, train_acc = train_one_epoch(
        model, train_loader_64, criterion, optimizer, device
    )

    val_loss, val_acc = evaluate(
        model, val_loader_64, criterion, device
    )

    wandb.log({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_accuracy": train_acc,
        "val_loss": val_loss,
        "val_accuracy": val_acc
    })

    print(
        f"Epoch {epoch+1}/10 | "
        f"Train Acc: {train_acc:.2f}% | "
        f"Val Acc: {val_acc:.2f}%"
    )

wandb.finish()

Epoch 1/10 | Train Acc: 41.73% | Val Acc: 48.08%
Epoch 2/10 | Train Acc: 56.29% | Val Acc: 57.97%
Epoch 3/10 | Train Acc: 62.13% | Val Acc: 56.08%
Epoch 4/10 | Train Acc: 65.91% | Val Acc: 69.41%
Epoch 5/10 | Train Acc: 69.07% | Val Acc: 73.22%
Epoch 6/10 | Train Acc: 70.73% | Val Acc: 70.18%
Epoch 7/10 | Train Acc: 72.50% | Val Acc: 78.60%
Epoch 8/10 | Train Acc: 73.65% | Val Acc: 77.08%
Epoch 9/10 | Train Acc: 74.76% | Val Acc: 79.76%
Epoch 10/10 | Train Acc: 75.86% | Val Acc: 78.76%


epoch,▁▂▃▃▄▅▆▆▇█
train_accuracy,▁▄▅▆▇▇▇███
train_loss,█▅▄▃▂▂▂▁▁▁
val_accuracy,▁▃▃▆▇▆█▇██
val_loss,▇▆█▃▂▃▁▂▁▁
epoch,10
train_accuracy,75.86
train_loss,0.69921
val_accuracy,78.758
val_loss,0.61407


In [34]:
train_loader_64, val_loader_64 = create_loaders(64)

set_seed(42)

wandb.init(
    project="VGG6-CIFAR10-Assignment1",
    entity="cs25m518-indian-institute-of-technology-madras",
    name="epochs-15-lr-0.001",
    config={
        "activation": "ReLU",
        "optimizer": "Adam",
        "learning_rate": 0.001,
        "batch_size": 64,
        "epochs": 15,
        "seed": 42
    }
)

model = vgg(
    cfg_vgg6,
    num_classes=10,
    batch_norm=True,
    activation="relu"
).to(device)

optimizer = get_optimizer("Adam", model, learning_rate=0.001)
criterion = nn.CrossEntropyLoss()

epoch_results = []

for epoch in range(15):
    train_loss, train_acc = train_one_epoch(
        model, train_loader_64, criterion, optimizer, device
    )

    val_loss, val_acc = evaluate(
        model, val_loader_64, criterion, device
    )

    wandb.log({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_accuracy": train_acc,
        "val_loss": val_loss,
        "val_accuracy": val_acc
    })

    epoch_results.append({
        "epoch": epoch + 1,
        "train_accuracy": train_acc,
        "val_accuracy": val_acc,
        "train_loss": train_loss,
        "val_loss": val_loss
    })

    print(
        f"Epoch {epoch+1}/15 | "
        f"Train Acc: {train_acc:.2f}% | "
        f"Val Acc: {val_acc:.2f}%"
    )

wandb.finish()

Epoch 1/15 | Train Acc: 45.24% | Val Acc: 51.27%
Epoch 2/15 | Train Acc: 56.51% | Val Acc: 60.62%
Epoch 3/15 | Train Acc: 61.18% | Val Acc: 59.33%
Epoch 4/15 | Train Acc: 64.48% | Val Acc: 67.50%
Epoch 5/15 | Train Acc: 66.46% | Val Acc: 70.89%
Epoch 6/15 | Train Acc: 68.23% | Val Acc: 64.33%
Epoch 7/15 | Train Acc: 69.83% | Val Acc: 70.31%
Epoch 8/15 | Train Acc: 71.21% | Val Acc: 73.72%
Epoch 9/15 | Train Acc: 72.32% | Val Acc: 72.57%
Epoch 10/15 | Train Acc: 73.09% | Val Acc: 76.91%
Epoch 11/15 | Train Acc: 74.07% | Val Acc: 76.27%
Epoch 12/15 | Train Acc: 74.08% | Val Acc: 74.07%
Epoch 13/15 | Train Acc: 74.75% | Val Acc: 79.55%
Epoch 14/15 | Train Acc: 75.70% | Val Acc: 79.39%
Epoch 15/15 | Train Acc: 75.99% | Val Acc: 81.16%


epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
train_accuracy,▁▄▅▅▆▆▇▇▇▇█████
train_loss,█▆▄▄▃▃▂▂▂▂▂▁▁▁▁
val_accuracy,▁▃▃▅▆▄▅▆▆▇▇▆███
val_loss,█▆▇▄▃▆▄▃▃▂▂▃▁▂▁
epoch,15
train_accuracy,75.99
train_loss,0.68933
val_accuracy,81.156
val_loss,0.55174


## 9. Q3: W&B Hyperparameter Sweep and Visualizations

The sweep explores activation functions, optimizers, learning rates, and batch sizes. Eight configurations are sampled using a random search strategy to identify promising hyperparameter combinations.

In [35]:
sweep_config = {
    "method": "random",
    "metric": {
        "name": "val_accuracy",
        "goal": "maximize"
    },
    "parameters": {
        "activation": {
            "values": ["relu", "silu", "gelu"]
        },
        "optimizer": {
            "values": ["Adam", "Nadam"]
        },
        "learning_rate": {
            "values": [0.001, 0.01]
        },
        "batch_size": {
            "values": [32, 64]
        },
        "epochs": {
            "value": 10
        },
        "seed": {
            "value": 42
        }
    }
}

sweep_id = wandb.sweep(
    sweep_config,
    project="VGG6-CIFAR10-Assignment1",
    entity="cs25m518-indian-institute-of-technology-madras"
)

print("Sweep ID:", sweep_id)

Create sweep with ID: pjyaw4dz
Sweep URL: https://wandb.ai/cs25m518-indian-institute-of-technology-madras/VGG6-CIFAR10-Assignment1/sweeps/pjyaw4dz
Sweep ID: pjyaw4dz


In [36]:
def sweep_train():
    with wandb.init() as run:
        config = wandb.config

        set_seed(config.seed)

        train_loader, val_loader = create_loaders(
            config.batch_size
        )

        model = vgg(
            cfg_vgg6,
            num_classes=10,
            batch_norm=True,
            activation=config.activation
        ).to(device)

        optimizer = get_optimizer(
            config.optimizer,
            model,
            learning_rate=config.learning_rate
        )

        criterion = nn.CrossEntropyLoss()

        for epoch in range(config.epochs):
            train_loss, train_acc = train_one_epoch(
                model, train_loader, criterion, optimizer, device
            )

            val_loss, val_acc = evaluate(
                model, val_loader, criterion, device
            )

            wandb.log({
                "epoch": epoch + 1,
                "train_loss": train_loss,
                "train_accuracy": train_acc,
                "val_loss": val_loss,
                "val_accuracy": val_acc
            })

            print(
                f"Epoch {epoch+1}/{config.epochs} | "
                f"Train Acc: {train_acc:.2f}% | "
                f"Val Acc: {val_acc:.2f}%"
            )

In [37]:
wandb.agent(
    sweep_id,
    function=sweep_train,
    count=8,
    project="VGG6-CIFAR10-Assignment1"
)

wandb: Agent Starting Run: 2m6q0pts with config:
wandb: 	activation: silu
wandb: 	batch_size: 32
wandb: 	epochs: 10
wandb: 	learning_rate: 0.001
wandb: 	optimizer: Adam
wandb: 	seed: 42
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


Epoch 1/10 | Train Acc: 45.13% | Val Acc: 51.04%
Epoch 2/10 | Train Acc: 55.82% | Val Acc: 61.07%
Epoch 3/10 | Train Acc: 60.81% | Val Acc: 65.07%
Epoch 4/10 | Train Acc: 64.31% | Val Acc: 68.25%
Epoch 5/10 | Train Acc: 66.40% | Val Acc: 72.77%
Epoch 6/10 | Train Acc: 68.21% | Val Acc: 74.06%
Epoch 7/10 | Train Acc: 69.77% | Val Acc: 76.09%
Epoch 8/10 | Train Acc: 70.71% | Val Acc: 76.47%
Epoch 9/10 | Train Acc: 72.29% | Val Acc: 77.45%
Epoch 10/10 | Train Acc: 72.94% | Val Acc: 79.48%


epoch,▁▂▃▃▄▅▆▆▇█
train_accuracy,▁▄▅▆▆▇▇▇██
train_loss,█▅▄▃▃▂▂▂▁▁
val_accuracy,▁▃▄▅▆▇▇▇▇█
val_loss,█▆▅▄▃▂▂▂▂▁
epoch,10
train_accuracy,72.94
train_loss,0.78219
val_accuracy,79.484
val_loss,0.59693


wandb: Agent Starting Run: u5ggj317 with config:
wandb: 	activation: relu
wandb: 	batch_size: 32
wandb: 	epochs: 10
wandb: 	learning_rate: 0.01
wandb: 	optimizer: Nadam
wandb: 	seed: 42
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


Epoch 1/10 | Train Acc: 44.17% | Val Acc: 56.15%
Epoch 2/10 | Train Acc: 58.51% | Val Acc: 63.48%
Epoch 3/10 | Train Acc: 64.46% | Val Acc: 65.22%
Epoch 4/10 | Train Acc: 67.82% | Val Acc: 73.85%
Epoch 5/10 | Train Acc: 70.39% | Val Acc: 75.09%
Epoch 6/10 | Train Acc: 72.16% | Val Acc: 78.94%
Epoch 7/10 | Train Acc: 73.70% | Val Acc: 77.88%
Epoch 8/10 | Train Acc: 74.63% | Val Acc: 77.29%
Epoch 9/10 | Train Acc: 76.05% | Val Acc: 76.34%
Epoch 10/10 | Train Acc: 76.38% | Val Acc: 81.10%


epoch,▁▂▃▃▄▅▆▆▇█
train_accuracy,▁▄▅▆▇▇▇███
train_loss,█▅▄▃▂▂▂▁▁▁
val_accuracy,▁▃▄▆▆▇▇▇▇█
val_loss,█▆▆▃▂▁▂▂▃▁
epoch,10
train_accuracy,76.376
train_loss,0.6787
val_accuracy,81.102
val_loss,0.57003


... (14 outputs left out)

epoch,▁▂▃▃▄▅▆▆▇█
train_accuracy,▁▄▅▆▇▇▇███
train_loss,█▅▄▃▃▂▂▁▁▁
val_accuracy,▁▂▅▄▅▆▇▇██
val_loss,█▇▄▅▄▂▂▂▁▁
epoch,10
train_accuracy,74.296
train_loss,0.74283
val_accuracy,77.404
val_loss,0.67259


wandb: Agent Starting Run: tj2rz60p with config:
wandb: 	activation: relu
wandb: 	batch_size: 32
wandb: 	epochs: 10
wandb: 	learning_rate: 0.001
wandb: 	optimizer: Nadam
wandb: 	seed: 42
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


Epoch 1/10 | Train Acc: 46.88% | Val Acc: 56.63%
Epoch 2/10 | Train Acc: 58.18% | Val Acc: 57.46%
Epoch 3/10 | Train Acc: 62.92% | Val Acc: 58.26%
Epoch 4/10 | Train Acc: 65.72% | Val Acc: 70.46%
Epoch 5/10 | Train Acc: 68.14% | Val Acc: 73.57%
Epoch 6/10 | Train Acc: 69.88% | Val Acc: 73.83%
Epoch 7/10 | Train Acc: 71.15% | Val Acc: 75.38%
Epoch 8/10 | Train Acc: 71.99% | Val Acc: 77.48%
Epoch 9/10 | Train Acc: 73.29% | Val Acc: 77.96%
Epoch 10/10 | Train Acc: 73.79% | Val Acc: 78.47%


epoch,▁▂▃▃▄▅▆▆▇█
train_accuracy,▁▄▅▆▇▇▇███
train_loss,█▅▄▃▃▂▂▁▁▁
val_accuracy,▁▁▂▅▆▇▇███
val_loss,▇██▃▂▂▂▁▁▁
epoch,10
train_accuracy,73.79
train_loss,0.75616
val_accuracy,78.47
val_loss,0.63093


## 10. Q4: Best Configuration and Reproduction

The best configuration identified through the W&B sweep is retrained using the selected hyperparameters. The checkpoint corresponding to the highest validation accuracy is saved.

In [39]:
# Q4: Reproduce the best configuration found by the W&B sweep

set_seed(42)

q4_config = {
    "activation": "silu",
    "optimizer": "Adam",
    "learning_rate": 0.01,
    "batch_size": 64,
    "epochs": 10,
    "seed": 42
}

train_loader, val_loader = create_loaders(q4_config["batch_size"])

model = vgg(
    cfg_vgg6,
    num_classes=10,
    batch_norm=True,
    activation=q4_config["activation"]
).to(device)

optimizer = get_optimizer(
    q4_config["optimizer"],
    model,
    learning_rate=q4_config["learning_rate"]
)

criterion = nn.CrossEntropyLoss()
best_val_accuracy = 0.0
checkpoint_path = "best_vgg6_q4.pth"

with wandb.init(
    project="VGG6-CIFAR10-Assignment1",
    name="Q4-best-config-reproduction",
    config=q4_config
) as run:

    for epoch in range(q4_config["epochs"]):

        train_loss, train_acc = train_one_epoch(
            model, train_loader, criterion, optimizer, device
        )

        val_loss, val_acc = evaluate(
            model, val_loader, criterion, device
        )

        wandb.log({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "train_accuracy": train_acc,
            "val_loss": val_loss,
            "val_accuracy": val_acc
        })

        print(
            f"Epoch {epoch + 1}/10 | "
            f"Train Acc: {train_acc:.2f}% | "
            f"Val Acc: {val_acc:.2f}%"
        )

        if val_acc > best_val_accuracy:
            best_val_accuracy = val_acc

            torch.save({
                "model_state_dict": model.state_dict(),
                "config": q4_config,
                "best_val_accuracy": best_val_accuracy,
                "epoch": epoch + 1
            }, checkpoint_path)

    artifact = wandb.Artifact(
        "vgg6-q4-best-model",
        type="model"
    )
    artifact.add_file(checkpoint_path)
    run.log_artifact(artifact)

print(f"\nBest validation accuracy: {best_val_accuracy:.2f}%")
print(f"Checkpoint saved: {checkpoint_path}")

Epoch 1/10 | Train Acc: 41.11% | Val Acc: 50.04%
Epoch 2/10 | Train Acc: 55.26% | Val Acc: 56.91%
Epoch 3/10 | Train Acc: 61.98% | Val Acc: 64.64%
Epoch 4/10 | Train Acc: 66.43% | Val Acc: 72.61%
Epoch 5/10 | Train Acc: 69.35% | Val Acc: 75.84%
Epoch 6/10 | Train Acc: 71.49% | Val Acc: 77.07%
Epoch 7/10 | Train Acc: 73.53% | Val Acc: 79.74%
Epoch 8/10 | Train Acc: 74.54% | Val Acc: 81.70%
Epoch 9/10 | Train Acc: 76.06% | Val Acc: 80.97%
Epoch 10/10 | Train Acc: 76.87% | Val Acc: 83.95%


epoch,▁▂▃▃▄▅▆▆▇█
train_accuracy,▁▄▅▆▇▇▇███
train_loss,█▅▄▃▃▂▂▁▁▁
val_accuracy,▁▂▄▆▆▇▇█▇█
val_loss,█▇▅▃▃▂▂▁▂▁
epoch,10
train_accuracy,76.87
train_loss,0.66631
val_accuracy,83.95
val_loss,0.46786



Best validation accuracy: 83.95%
Checkpoint saved: best_vgg6_q4.pth


In [40]:
import os
import torch

checkpoint_path = "best_vgg6_q4.pth"

print("Checkpoint exists:", os.path.exists(checkpoint_path))

checkpoint = torch.load(
    checkpoint_path,
    map_location=device,
    weights_only=False
)

print("Saved epoch:", checkpoint["epoch"])
print("Saved validation accuracy:", checkpoint["best_val_accuracy"])
print("Saved configuration:", checkpoint["config"])

Checkpoint exists: True
Saved epoch: 10
Saved validation accuracy: 83.95
Saved configuration: {'activation': 'silu', 'optimizer': 'Adam', 'learning_rate': 0.01, 'batch_size': 64, 'epochs': 10, 'seed': 42}
